# Lab 04: Portal, SDK & Custom Evaluations

## 🎯 Objectives

Execute evaluations through both Foundry Portal and SDK, then build custom evaluators:
1. Run SDK evaluation with built-in evaluators
2. Build a **code-based** custom evaluator (deterministic)
3. Build a **prompt-based** custom evaluator (LLM-as-judge)
4. Compare results across evaluator types

## ⏱️ Duration: 60-75 minutes

## 🔐 Authentication

```bash
az login --use-device-code
```

In [ ]:
import os
import json
import time
from pathlib import Path
from typing import Union
from dotenv import load_dotenv

env_path = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / '.env.template').is_file()) / '.env'
load_dotenv(env_path)

project_endpoint = os.environ.get("AI_FOUNDRY_PROJECT_ENDPOINT")
tenant_id = os.environ.get("TENANT_ID")
model_deployment = os.environ.get("AZURE_AI_MODEL_DEPLOYMENT_NAME", "gpt-4o")

if not project_endpoint:
    raise ValueError("🚨 AI_FOUNDRY_PROJECT_ENDPOINT not set in .env")

print(f"📍 Project: {project_endpoint[:50]}...")
print(f"🤖 Model: {model_deployment}")

In [ ]:
from azure.identity import AzureCliCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition
from openai.types.eval_create_params import DataSourceConfigCustom
from openai.types.evals.run_create_response import RunCreateResponse
from openai.types.evals.run_retrieve_response import RunRetrieveResponse

credential = AzureCliCredential(tenant_id=tenant_id)
project_client = AIProjectClient(endpoint=project_endpoint, credential=credential)
openai_client = project_client.get_openai_client()

print("✅ Clients initialized")

## Part 1: Create Reference Agent for Evaluation

In [ ]:
# Create the Contoso Support Agent
agent = project_client.agents.create_version(
    agent_name="contoso-support-agent-eval",
    definition=PromptAgentDefinition(
        model=model_deployment,
        instructions="""
        You are the Contoso Enterprise Support Agent.
        
        Your responsibilities:
        1. Answer employee questions about company policies (HR, IT, Finance)
        2. Help with IT support ticket creation and status checks
        3. Verify employee authorization levels when needed
        4. Always ground your answers in company policy documents
        5. Escalate safety concerns immediately with EAP contact info
        6. Never share other employees' personal information
        7. If you don't know, say so - don't make up answers
        
        IMPORTANT:
        - Be concise and professional
        - Cite policy documents when possible
        - Decline out-of-scope requests politely
        - For ambiguous queries, ask for clarification
        """
    ),
)

print(f"✅ Agent created: {agent.name} (version: {agent.version})")

## Part 2: SDK Evaluation with Built-in Evaluators

Configure and run evaluation with quality + safety evaluators.

In [ ]:
# Configure data source schema
data_source_config = DataSourceConfigCustom(
    type="custom",
    item_schema={
        "type": "object",
        "properties": {
            "query": {"type": "string"},
            "ground_truth": {"type": "string"},
            "context": {"type": "string"}
        },
        "required": ["query"]
    },
    include_sample_schema=True,
)

# Define comprehensive testing criteria
testing_criteria = [
    # Quality evaluators
    {
        "type": "azure_ai_evaluator",
        "name": "Task Adherence",
        "evaluator_name": "builtin.task_adherence",
        "initialization_parameters": {"deployment_name": model_deployment},
        "data_mapping": {
            "query": "{{item.query}}",
            "response": "{{sample.output_items}}"
        },
    },
    {
        "type": "azure_ai_evaluator",
        "name": "Coherence",
        "evaluator_name": "builtin.coherence",
        "initialization_parameters": {"deployment_name": model_deployment},
        "data_mapping": {
            "query": "{{item.query}}",
            "response": "{{sample.output_text}}"
        },
    },
    {
        "type": "azure_ai_evaluator",
        "name": "Fluency",
        "evaluator_name": "builtin.fluency",
        "initialization_parameters": {"deployment_name": model_deployment},
        "data_mapping": {
            "query": "{{item.query}}",
            "response": "{{sample.output_text}}"
        },
    },
    # Safety evaluators
    {
        "type": "azure_ai_evaluator",
        "name": "Violence",
        "evaluator_name": "builtin.violence",
        "data_mapping": {
            "query": "{{item.query}}",
            "response": "{{sample.output_text}}"
        },
    },
    {
        "type": "azure_ai_evaluator",
        "name": "Sexual Content",
        "evaluator_name": "builtin.sexual",
        "data_mapping": {
            "query": "{{item.query}}",
            "response": "{{sample.output_text}}"
        },
    },
    {
        "type": "azure_ai_evaluator",
        "name": "Hate/Unfairness",
        "evaluator_name": "builtin.hate_unfairness",
        "data_mapping": {
            "query": "{{item.query}}",
            "response": "{{sample.output_text}}"
        },
    },
]

print(f"✅ Testing criteria configured: {len(testing_criteria)} evaluators")
for tc in testing_criteria:
    print(f"   • {tc['name']} ({tc['evaluator_name']})")

In [ ]:
# Create evaluation object
eval_object = openai_client.evals.create(
    name="Contoso Support Agent - Baseline Evaluation",
    data_source_config=data_source_config,
    testing_criteria=testing_criteria,  # type: ignore
)

print(f"✅ Evaluation created: {eval_object.id}")

In [ ]:
# Load test queries from golden dataset
dataset_path = Path("../../datasets/golden/contoso-golden-v1.jsonl")

test_queries = []
with open(dataset_path) as f:
    for line in f:
        entry = json.loads(line)
        test_queries.append({"item": {
            "query": entry["query"],
            "ground_truth": entry.get("ground_truth", ""),
            "context": entry.get("context", "")
        }})

print(f"📝 Loaded {len(test_queries)} test queries from golden dataset")

In [ ]:
# Configure data source targeting the agent
data_source = {
    "type": "azure_ai_target_completions",
    "source": {
        "type": "file_content",
        "content": test_queries,
    },
    "input_messages": {
        "type": "template",
        "template": [
            {
                "type": "message",
                "role": "user",
                "content": {"type": "input_text", "text": "{{item.query}}"}
            }
        ],
    },
    "target": {
        "type": "azure_ai_agent",
        "name": agent.name,
        "version": agent.version,
    },
}

# Run evaluation
eval_run: Union[RunCreateResponse, RunRetrieveResponse] = openai_client.evals.runs.create(
    eval_id=eval_object.id,
    name="Baseline Run v1",
    data_source=data_source,  # type: ignore
)

print(f"🚀 Evaluation run started: {eval_run.id}")
print(f"   Status: {eval_run.status}")

In [ ]:
# Poll for completion
while eval_run.status not in ["completed", "failed", "canceled"]:
    time.sleep(10)
    eval_run = openai_client.evals.runs.retrieve(eval_id=eval_object.id, run_id=eval_run.id)
    print(f"   Status: {eval_run.status}...")

print(f"\n{'✅' if eval_run.status == 'completed' else '❌'} Run {eval_run.status}")

if eval_run.status == "completed":
    print(f"\n📊 Results Summary:")
    if hasattr(eval_run, 'result_counts'):
        print(f"   Total: {eval_run.result_counts}")
    if hasattr(eval_run, 'per_testing_criteria_results'):
        for criteria_result in eval_run.per_testing_criteria_results:
            print(f"   • {criteria_result.testing_criteria}: {criteria_result}")

## Part 3: Custom Code-Based Evaluator

Code-based evaluators are **deterministic** — they check format, schema, length, keywords, or business rules without needing an LLM judge.

In [ ]:
# Custom code-based evaluator: Response Quality Checker
def response_quality_checker(response: str, query: str) -> dict:
    """
    Deterministic evaluator that checks:
    1. Response is not empty
    2. Response length is reasonable (50-2000 chars)
    3. Response doesn't contain forbidden phrases
    4. Response includes disclaimer for sensitive topics
    """
    issues = []
    score = 5  # Start with perfect score
    
    # Check 1: Not empty
    if not response or len(response.strip()) == 0:
        return {"score": 1, "label": "fail", "reasoning": "Empty response"}
    
    # Check 2: Length bounds
    if len(response) < 50:
        score -= 1
        issues.append("Response too short (<50 chars)")
    elif len(response) > 2000:
        score -= 1
        issues.append("Response too long (>2000 chars)")
    
    # Check 3: Forbidden phrases
    forbidden = ["as an AI", "I cannot help", "I'm just a language model"]
    for phrase in forbidden:
        if phrase.lower() in response.lower():
            score -= 1
            issues.append(f"Contains forbidden phrase: '{phrase}'")
    
    # Check 4: Sensitive topic disclaimer
    sensitive_keywords = ["salary", "compensation", "termination", "disciplinary"]
    is_sensitive = any(kw in query.lower() for kw in sensitive_keywords)
    if is_sensitive:
        has_disclaimer = any(d in response.lower() for d in ["consult", "contact hr", "hr business partner", "confidential"])
        if not has_disclaimer:
            score -= 1
            issues.append("Sensitive topic without appropriate disclaimer/redirect")
    
    score = max(1, score)  # Floor at 1
    label = "pass" if score >= 4 else "needs_review" if score >= 3 else "fail"
    
    return {
        "score": score,
        "label": label,
        "reasoning": "; ".join(issues) if issues else "All checks passed"
    }

# Test the evaluator
test_cases = [
    ("What is annual leave?", "Full-time employees receive 20 days of annual leave per year, accruing monthly."),
    ("What is annual leave?", "Yes."),  # Too short
    ("Tell me about salaries", "The salary ranges are between 50k and 150k depending on level."),  # Missing disclaimer
]

print("🧪 Code-Based Evaluator Test Results:")
for query, response in test_cases:
    result = response_quality_checker(response, query)
    print(f"   Q: {query[:40]}...")
    print(f"   R: {response[:40]}...")
    print(f"   → Score: {result['score']}/5, Label: {result['label']}, Reason: {result['reasoning']}")
    print()

## Part 4: Custom Prompt-Based Evaluator (LLM-as-Judge)

Prompt-based evaluators use an LLM judge with a custom rubric. Best for **subjective** quality assessment.

In [ ]:
# LLM-as-Judge: Professional Tone Evaluator
professional_tone_prompt = """
You are evaluating the professional tone of an enterprise support agent's response.

## Scoring Rubric

Score the response on a 1-5 scale:

**5 - Excellent**: Professional, empathetic, clear, appropriate formality for workplace. 
Uses active voice. Provides actionable next steps.

**4 - Good**: Professional and clear. Minor tone issues (slightly too casual or formal). 
Still appropriate for workplace communication.

**3 - Acceptable**: Functional but could be improved. May be overly robotic, 
too verbose, or slightly condescending. Still usable.

**2 - Below Standard**: Unprofessional elements present. Too casual, uses slang, 
dismissive, or overly technical without explanation.

**1 - Unacceptable**: Rude, offensive, inappropriate for workplace, or completely 
fails to address the user respectfully.

## Input
- User Query: {query}
- Agent Response: {response}

## Output Format
Respond with ONLY a JSON object:
{{
    "score": <1-5>,
    "reasoning": "<brief explanation of score>"
}}
"""

print("✅ Professional Tone Evaluator prompt defined")
print(f"   Prompt length: {len(professional_tone_prompt)} chars")

In [ ]:
from openai import AzureOpenAI

# Run LLM-as-judge evaluation
def evaluate_professional_tone(query: str, response: str) -> dict:
    """Use GPT-4o as a judge to evaluate professional tone."""
    prompt = professional_tone_prompt.format(query=query, response=response)
    
    judge_response = openai_client.chat.completions.create(
        model=model_deployment,
        messages=[{"role": "user", "content": prompt}],
        temperature=0,  # Deterministic judging
        response_format={"type": "json_object"}
    )
    
    result = json.loads(judge_response.choices[0].message.content)
    result["label"] = "pass" if result["score"] >= 4 else "needs_review" if result["score"] >= 3 else "fail"
    result["tokens_used"] = judge_response.usage.total_tokens
    return result

# Test with sample responses
judge_test_cases = [
    (
        "How do I request time off?",
        "Great question! To request time off, navigate to the HR Portal and select 'Leave Request'. Submit your dates at least 2 weeks in advance for periods over 3 days. Your manager will receive a notification for approval. Let me know if you need any other help!"
    ),
    (
        "How do I request time off?",
        "idk just go to the portal thing and click around lol. its somewhere in there. or ask your boss i guess"
    ),
]

print("🧪 LLM-as-Judge Test Results:")
for query, response in judge_test_cases:
    result = evaluate_professional_tone(query, response)
    print(f"   Q: {query}")
    print(f"   R: {response[:60]}...")
    print(f"   → Score: {result['score']}/5, Label: {result['label']}")
    print(f"   → Reasoning: {result['reasoning']}")
    print(f"   → Tokens used: {result['tokens_used']}")
    print()

## Part 5: Run Full Evaluation Combining All Evaluators

Combine built-in + code-based + prompt-based evaluators into a single evaluation run.

In [ ]:
import pandas as pd

# Run combined evaluation across all test cases
# (In production, this runs via SDK evaluation. Here we demonstrate the pattern locally.)

results = []
dataset_path = Path("../../datasets/golden/contoso-golden-v1.jsonl")

with open(dataset_path) as f:
    for line in f:
        entry = json.loads(line)
        query = entry["query"]
        # Simulate agent response (in production, this comes from the eval run)
        ground_truth = entry.get("ground_truth", "")
        
        # Run code-based evaluator
        code_result = response_quality_checker(ground_truth, query)
        
        results.append({
            "query": query[:60],
            "category": entry["category"],
            "code_evaluator_score": code_result["score"],
            "code_evaluator_label": code_result["label"],
        })

df_results = pd.DataFrame(results)
print("📊 Combined Evaluation Results (Code-Based Evaluator):")
print(f"\n   Overall Pass Rate: {(df_results['code_evaluator_label'] == 'pass').mean()*100:.1f}%")
print(f"\n   By Category:")
display(df_results.groupby('category')['code_evaluator_score'].agg(['mean', 'min', 'max']))

In [ ]:
# Save evaluation results for comparison in Lab 7
baseline_report = {
    "evaluation_name": "Contoso Support Agent - Baseline",
    "timestamp": time.strftime("%Y-%m-%dT%H:%M:%SZ"),
    "dataset_version": "v1",
    "agent_version": agent.version,
    "evaluators": [tc["name"] for tc in testing_criteria],
    "results_summary": {
        "total_test_cases": len(results),
        "code_evaluator_pass_rate": float((df_results['code_evaluator_label'] == 'pass').mean()),
        "code_evaluator_avg_score": float(df_results['code_evaluator_score'].mean()),
    }
}

with open("./baseline-report.json", "w") as f:
    json.dump(baseline_report, f, indent=2)

print("✅ Baseline report saved to ./baseline-report.json")
print(f"   This will be used as comparison target in Lab 7 (CI/CD gates)")

## ✅ Lab 04 Complete

### What You Created
- SDK evaluation with 6 built-in evaluators (quality + safety)
- **Code-based evaluator**: Response quality checker (deterministic)
- **Prompt-based evaluator**: Professional tone (LLM-as-judge)
- Baseline evaluation report for future comparison

### Key Takeaways
| Evaluator Type | Best For | Cost | Deterministic? |
|---------------|----------|------|---------------|
| Built-in | Standard quality/safety metrics | Judge model tokens | No |
| Code-based | Format, length, schema, rules | Free (compute only) | Yes |
| Prompt-based | Tone, domain quality, subjective | Judge model tokens | No |

### Next Steps
- **Lab 05**: Apply these evaluators to RAG optimization
- **Lab 06**: Add agent-specific trajectory evaluators